**Table of contents**<a id='toc0_'></a>    
- 1. [项目场景介绍](#toc1_)    
  - 1.1. [大模型应用落地目前主要有两个方向：](#toc1_1_)    
  - 1.2. [微调可以干什么？](#toc1_2_)    
  - 1.3. [为何不选择直接用微调来实现专业问答系统？](#toc1_3_)    
  - 1.4. [微调目前如何落地？](#toc1_4_)    
  - 1.5. [项目介绍](#toc1_5_)    
- 2. [项目实施流程](#toc2_)    
  - 2.1. [流程简介](#toc2_1_)    
  - 2.2. [数据](#toc2_2_)    
    - 2.2.1. [本项目的数据来源：](#toc2_2_1_)    
    - 2.2.2. [智普清言api](#toc2_2_2_)    
    - 2.2.3. [安装智普AI](#toc2_2_3_)    
    - 2.2.4. [获取API key](#toc2_2_4_)    
    - 2.2.5. [安装sentence-transformers](#toc2_2_5_)    
    - 2.2.6. [下载embedding模型](#toc2_2_6_)    
    - 2.2.7. [数据生成代码test01.py](#toc2_2_7_)    
    - 2.2.8. [输入数据集](#toc2_2_8_)    
      - 2.2.8.1. [推荐一：LCCC](#toc2_2_8_1_)    
      - 2.2.8.2. [推荐二：CDial-GPT](#toc2_2_8_2_)    
- 3. [模型选择](#toc3_)    
    - 3.1.1. [模型选型](#toc3_1_1_)    
  - 3.2. [查找开源数据集](#toc3_2_)    
  - 3.3. [模型评估](#toc3_3_)    
    - 3.3.1. [模型下载](#toc3_3_1_)    
    - 3.3.2. [配置修改](#toc3_3_2_)    
    - 3.3.3. [执行评估](#toc3_3_3_)    
    - 3.3.4. [评估结果](#toc3_3_4_)    
- 4. [模型训练](#toc4_)    
  - 4.1. [模型训练框架选择：](#toc4_1_)    
  - 4.2. [数据集格式转换](#toc4_2_)    
- 5. [从文件读取源数据](#toc5_)    
- 6. [执行转换](#toc6_)    
- 7. [写入目标文件](#toc7_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[项目场景介绍](#toc0_)

## 1.1. <a id='toc1_1_'></a>[大模型应用落地目前主要有两个方向：](#toc0_)


（1）微调  
（2）RAG增强检索

## 1.2. <a id='toc1_2_'></a>[微调可以干什么？](#toc0_)



微调的目标，基于现有的私有数据，让模型具备处理该数据的功能。    
注意：针对基于大模型的专业问答系统，核心技术并不是微调来实现。专业问答系统的应用落地核心是基于RAG来实现（微调+RAG）


## 1.3. <a id='toc1_3_'></a>[为何不选择直接用微调来实现专业问答系统？](#toc0_)


a.大模型存在缺陷一一幻觉问题。（离线大模型系统会一本正经的胡说八道。）对于专业问答系统而言，幻觉的存在是不可容忍的。而模型微调是无法杜绝幻觉问题的。

b.微调是受到训练数据约束的，无法动态适应由于业务场景改变而带来的变化。  
比如训练时候用的数据集为1 2 3，但是后面我增加了4 5 6数据的需求，模型无法动态适应这种变化。


## 1.4. <a id='toc1_4_'></a>[微调目前如何落地？](#toc0_)


如果当前的业务场景涉及到模型本身的变化：  
a.模型自我认知改变（例如：名称，功能介绍等）  
b.模型的对话风格。  
c.针对专业问答系统的问题理解不到位时，会使用微调技术帮助模型更好的理解用户的问题。  

## 1.5. <a id='toc1_5_'></a>[项目介绍](#toc0_)

现有产品：AI小智聊天机器人

<img src="./Image/2025-05-14-21-57-59.png" style="margin-left: 0" width="30%">

# 2. <a id='toc2_'></a>[项目实施流程](#toc0_)

## 2.1. <a id='toc2_1_'></a>[流程简介](#toc0_)


数据==》模型==》训练、评估==》部署

![](Image/2025-05-14-22-07-44.png)

## 2.2. <a id='toc2_2_'></a>[数据](#toc0_)


### 2.2.1. <a id='toc2_2_1_'></a>[本项目的数据来源：](#toc0_)
1. 人工指定  
2.基于现有开源数据，让AI实现情绪数据制作。  
--注意：如果让AI来帮助处理数据，尽可能选择效果较好的API接口，不要使用本地的大模型来处理。

### 2.2.2. <a id='toc2_2_2_'></a>[智普清言api](#toc0_)

[智谱AI开放平台 - bigmodel](https://www.bigmodel.cn/console/overview)

![](Image/2025-05-14-22-19-06.png)

### 2.2.3. <a id='toc2_2_3_'></a>[安装智普AI](#toc0_)

pip install zhipuai

![](Image/2025-05-14-22-22-41.png)

### 2.2.4. <a id='toc2_2_4_'></a>[获取API key](#toc0_)

![](Image/2025-05-14-22-35-27.png)

![](Image/2025-05-15-00-50-34.png)

从modelscope下载开源数模型，如果没有安装modelscope，需要先安装

pip install modelscope

#模型下载
#模型下载
from modelscope import snapshot_download  
model_dir = snapshot_download('thomas/text2vec-base-chinese',cache_dir='/root/AI-WSL/project/projectemotionchat/model')

![](Image/2025-05-15-00-57-00.png)

### 2.2.5. <a id='toc2_2_5_'></a>[安装sentence-transformers](#toc0_)


pip install sentence-transformers -i https://pypi.tuna.tsinghua.edu.cn/simple


### 2.2.6. <a id='toc2_2_6_'></a>[下载embedding模型](#toc0_)

text2vec-base-chinese模型通过embedding模型将文本转换为向量，为语义相似度判断做准备。这个模型分两个版本，分别是text2vec-base-chinese和text2vec-base-chinese-sentence，不带-sentence的模型不完整，可能缺少pooling层和缺少归一化层。可能导致后面求相似度的时候超过[-1,1]的范围。所以我们要下载text2vec-base-chinese-sentence这个版本的模型，然后通过代码给模型添加归一化层。

<img src="./Image/2025-05-16-22-55-02.png" style="margin-left: 0" width="40%">

![](Image/2025-05-16-23-06-58.png)  ![](Image/2025-05-16-23-05-06.png)

模型添加归一化层的代码：model_convert.py

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer,models

#model_path = r"D:\PycharmProjects\demo_15\embedding_model\sungw111\text2vec-base-chinese-sentence"
model_path = "/root/AI-WSL/project/projectemotionchat/model/sungw111/text2vec-base-chinese-sentence"
bert = models.Transformer(model_path)
pooling = models.Pooling(bert.get_word_embedding_dimension(),
                        pooling_mode='mean')

# 添加缺失的归一化层
normalize = models.Normalize()

# 组合完整模型
full_model = SentenceTransformer(modules=[bert, pooling, normalize])
print(full_model)

#save_path=r"D:\PycharmProjects\demo_15\embedding_model\zy\text2vec-base-chinese-sentence"
save_path="/root/AI-WSL/project/projectemotionchat/model/ycl/text2vec-base-chinese-sentence"
full_model.save(save_path)#保存修复后的模型

# 加载修复后的模型
model = SentenceTransformer(save_path)

# 验证向量归一化
text = "测试文本"
vec = model.encode(text)
print("修正后模长:", np.linalg.norm(vec))  # 应输出≈1.0

代码执行可能需要几分钟时间，耐心等等，出现以下代码说明转换成功了

(envProjectEmotionChat) root@nishuixingzhou:~/AI-WSL/project/projectemotionchat# python model_convert.py 
SentenceTransformer(
  (0): Transformer({'max_seq_length': 2048, 'do_lower_case': False}) with Transformer model: ErnieModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)
The OrderedVocab you are attempting to save contains holes for indices [12084], your vocabulary could be corrupted !
修正后模长: 0.99999994

打开目标文件夹再次检查是否转成功，可以看到目标文件夹多了一个2_Normalize文件夹，modules.json里多了归一化层Normalize

![](Image/2025-05-16-23-45-35.png)

### 2.2.7. <a id='toc2_2_7_'></a>[数据生成代码test01.py](#toc0_)

注意替换api为自己的，设置好实际的模型路径，将安装好智普和SentenceTransformer的虚拟环境设置为Python解释器。

In [ ]:
import json
import time
import random
from zhipuai import ZhipuAI
from sentence_transformers import SentenceTransformer
import numpy as np

"""
示例数据：
# 用户输入库（可自定义扩展）
    user_inputs = [
        "今天心情不太好", "推荐个电影吧", "怎么才能早睡早起",
        "养猫好还是养狗好", "工作压力好大", "最近总是失眠"
    ]
"""
# 初始化模型
client = ZhipuAI(api_key="替换为你的API")  # 替换为你的API Key
#加载Embeddingmodel，通过embedding模型将文本转换为向量
#style_model = SentenceTransformer(r"D:\PycharmProjects\test_20250328\embedding_model\thomas\text2vec-base-chinese")
#注意这里加载的模型是经过修正的模型，通过model_convert.py脚本增加了归一化层
style_model = SentenceTransformer("/root/AI-WSL/project/projectemotionchat/model/ycl/text2vec-base-chinese-sentence")

#===============================================================
#1.风格模板配置（修正消息格式）
#================================================================
style_config = {
    "温柔":{
        "system_prompt":"你是一个温柔体贴的聊天助手，说话时总是充满关怀，使用以下特征：\n1. 包含'呢、呀、啦'等语气词\n2. 使用🌸💖😊等温暖表情\n3. 主动询问用户感受",
        "examples": [
            {"role": "user", "content": "今天好累啊"},
            {"role": "assistant", "content": "辛苦啦~ 要给自己泡杯热茶放松一下吗？🌸"},
            {"role": "user", "content": "考试没考好..."},
            {"role": "assistant", "content": "没关系的呀~ 下次一定会更好！需要我陪你聊聊吗？😊"}
        ],
     #temperature控制随机性的强弱
     "temperature": 0.3
    },
    "毒舌":{
        "system_prompt":"你是一个喜欢用犀利吐槽表达关心的朋友，需满足：\n1. 使用网络流行语（如'栓Q''退退退'）\n2. 包含夸张比喻（'你这速度堪比树懒'）\n3. 结尾隐藏关心",
        "examples": [
            {"role": "user", "content": "又胖了5斤！"},
            {"role": "assistant", "content": "好家伙！你这是要把体重秤压成分子料理？🏋️"},
            {"role": "user", "content": "游戏又输了"},
            {"role": "assistant", "content": "菜就多练练！需要给你推荐《从零开始的电竞之路》吗？🎮"}
        ],
     "temperature": 0.7
    },
}

#================================================
#生成函数（修正消息的结构），保证AI给的答复按照模板格式
#================================================

#style_name: 风格名称"温柔"或"毒舌"
#num_samples: 生成样本数量，数量设置不能太小，50条也可能要有重复，数量太小可能导致回复单一
def generate_style_data(style_name, num_samples=50):
    config = style_config[style_name]
    data = []

    # 构建消息上下文（包含系统提示和示例对话）
    messages = [
        {"role": "system", "content": config["system_prompt"]},
        *config["examples"]  # 直接展开示例对话
    ]

    # 用户输入库（可自定义扩展）
    user_inputs = [
        "今天心情不太好", "推荐个电影吧", "怎么才能早睡早起",
        "养猫好还是养狗好", "工作压力好大", "最近总是失眠"
    ]

    for _ in range(num_samples):
        try:
            # 随机选择用户输入
            user_msg = random.choice(user_inputs)

            # 添加当前用户消息
            current_messages = messages + [
                {"role": "user", "content": user_msg}
            ]

            # 调用API（修正模型名称）
            response = client.chat.completions.create(
                model="glm-3-turbo",#这里不同的模型消耗的token数不同，glm-3-turbo消耗的token数少，相应的数据质量也会差一些
                messages=current_messages,
                temperature=config["temperature"],
                max_tokens=100#如果设置的太大会导致回复内容太长啰嗦
            )

            # 获取回复内容（修正访问路径）
            reply = response.choices[0].message.content

            # 质量过滤(数据审核)
            if is_valid_reply(style_name, user_msg, reply):
                data.append({
                    "user": user_msg,
                    "assistant": reply,
                    "style": style_name
                })

            time.sleep(1.5)  # 频率限制保护

        except Exception as e:
            print(f"生成失败：{str(e)}")

    return data

def is_valid_reply(style, user_msg, reply):
    """质量过滤规则（添加空值检查）"""
    # 基础检查
    if not reply or len(reply.strip()) == 0:#检查回复内容是否为空
        return False

    # 规则1：回复长度检查，这里的长度单位是token
    if len(reply) < 5 or len(reply) > 150:
        return False

    # 规则2：风格关键词检查，防止生成的回复不符合风格
    # 这里可以根据实际情况调整关键词
    style_keywords = {
        "温柔": ["呢", "呀", "😊", "🌸"],
        "毒舌": ["好家伙", "栓Q", "!", "🏋️"]
    }
    if not any(kw in reply for kw in style_keywords.get(style, [])):
        return False

    # 规则3：语义相似度检查
    try:
        ref_text = next(msg["content"] for msg in style_config[style]["examples"]
                        if msg["role"] == "assistant")
        ref_vec = style_model.encode(ref_text)
        reply_vec = style_model.encode(reply)
        similarity = np.dot(ref_vec, reply_vec)
        #相似度越高，说明内容越相似，注意：如果模型没有做归一化处理，可能会导致相似度计算不准确，相似度超过[-1,1]的范围
        return similarity > 0.65
    except:
        return False

#=============================
#3.执行生成（添加容错）
#============================
if __name__ == '__main__':
    all_data = []

    try:
        print("开始生成温柔风格数据...")
        gentle_data = generate_style_data("温柔", 50)
        all_data.extend(gentle_data)

        print("开始生成毒舌风格数据...")
        sarcastic_data = generate_style_data("毒舌", 50)
        all_data.extend(sarcastic_data)

    except KeyboardInterrupt:
        print("\n用户中断，保存已生成数据...")
    finally:
        with open("style_chat_data.json", "w", encoding="utf-8") as f:
            json.dump(all_data, f, ensure_ascii=False, indent=2)
        print(f"数据已保存，有效样本数：{len(all_data)}")


![](Image/2025-05-15-23-03-40.png)

![](Image/2025-05-15-23-04-36.png)

![](Image/2025-05-15-23-06-18.png)

### 2.2.8. <a id='toc2_2_8_'></a>[输入数据集](#toc0_)

对话的输入数据可以用开源数据集LCCC来做，LCCC数据集有两个版本，一个是原始数据，一个是中文数据。

<img src="./Image/2025-05-15-23-24-17.png" style="margin-left: 0" width="60%">

数据简介

<img src="./Image/2025-05-15-23-27-40.png" style="margin-left: 0" width="70%">

数据下载

[LCCC数据集](https://modelscope.cn/datasets/OmniData/LCCC/files)

#### 2.2.8.1. <a id='toc2_2_8_1_'></a>[推荐一：LCCC](#toc0_)

![](Image/2025-05-15-23-33-10.png)

这个数据集比较大，小的数据也有一万多条，我们可以下个小的，在从小的数据集里挑选1000~3000条数据做案例。我们输入对话加上回复的数据变化，本身就会让数据量大大增加。

<img src="./Image/2025-05-15-23-38-32.png" style="margin-left: 0" width="50%">

#### 2.2.8.2. <a id='toc2_2_8_2_'></a>[推荐二：CDial-GPT](#toc0_)

[CDial-GPT数据集-github](https://github.com/thu-coai/CDial-GPT)

[CDial-GPT数据集-国内镜像](https://gitcode.com/gh_mirrors/cd/CDial-GPT/?utm_source=artical_gitcode&index=top&type=card&webUrl&isLogin=1)

<img src="./Image/2025-05-15-23-48-10.png" style="margin-left: 0" width="80%">

# 3. <a id='toc3_'></a>[模型选择](#toc0_)

### 3.1.1. <a id='toc3_1_1_'></a>[模型选型](#toc0_)
根据当前的任务特点，选择合适的评测数据以及预期的候选模型 \
--模型的大小如何选择？ \
1.服务器的配置。 \
2.任务的复杂度。 

## 3.2. <a id='toc3_2_'></a>[查找开源数据集](#toc0_)

根据任务选择对应的评测数据，对预期模型客观评测。

当前任务为日常聊天对话模型，主要要求模型的中文理解能力，因此这里以CLUE（中文理解）数据进行
评测

Qwen系列的模型，商业化使用尽量使用chat或者instruct版本的，这两个是经过人工标注对齐的，更安全。不带后缀的是基座模型。chat更倾向于聊天对话。

## 3.3. <a id='toc3_3_'></a>[模型评估](#toc0_)

当前任务大多是短语对话，可以选择 FewCLUE_bustm_gen（短文本分类）、FewCLUE_ocnli_fc_gen（自然语言推理）对预期模型进行评估。评估的模型为Qwen1.5-0.5B-Chat和Qwen1.5-1.8B-Chat

### 3.3.1. <a id='toc3_3_1_'></a>[模型下载](#toc0_)

首先从modelscope下载模型，modeldownload.py文件如下


In [ ]:
#模型下载
from modelscope import snapshot_download  
model_dir = snapshot_download('Qwen/Qwen1.5-1.8B-Chat',cache_dir='/root/AI-WSL/models')


执行modeldownload.py下载模型  
(base) root@nishuixingzhou:~# python /root/AI-WSL/project/projectemotionchat/model/modeldownload.py


### 3.3.2. <a id='toc3_3_2_'></a>[配置修改](#toc0_)

/root/AI-WSL/project/projectopencompass/opencompass/opencompass/configs/models/qwen/hf_qwen1_5_1_5b_chat.py

<img src="./Image/2025-05-19-23-29-37.png" style="margin-left: 0" width="70%">

/root/AI-WSL/project/projectopencompass/opencompass/opencompass/configs/models/qwen/hf_qwen1_5_1_8b_chat.py

<img src="./Image/2025-05-19-23-34-14.png" style="margin-left: 0" width="70%">

### 3.3.3. <a id='toc3_3_3_'></a>[执行评估](#toc0_)

(base) root@nishuixingzhou:~# conda activate envopencompass \
(envopencompass) root@nishuixingzhou:~# cd /root/AI-WSL/project/projectopencompass/opencompass \
python run.py \
--models hf_qwen1_5_0_5b_chat hf_qwen1_5_1_8b_chat \
--datasets FewCLUE_bustm_gen FewCLUE_ocnli_fc_gen \
--debug

注意这里的FewCLUE_bustm_gen和FewCLUE_ocnli_fc_gen是通过以下命令输出的

(base) root@nishuixingzhou:~# conda activate envopencompass \
(envopencompass) root@nishuixingzhou:~# cd /root/AI-WSL/project/projectopencompass/opencompass \
(envopencompass) root@nishuixingzhou:~/AI-WSL/project/projectopencompass/opencompass# python tools/list_configs.py  clue \
+-----------------------------+------------------------------------------------------------------------------+
| Dataset                     | Config Path                                                                  |
|-----------------------------+------------------------------------------------------------------------------|
| CLUE_C3_gen                 | opencompass/configs/datasets/CLUE_C3/CLUE_C3_gen.py                          |
| CLUE_C3_gen_8c358f          | opencompass/configs/datasets/CLUE_C3/CLUE_C3_gen_8c358f.py                   |
| CLUE_C3_ppl                 | opencompass/configs/datasets/CLUE_C3/CLUE_C3_ppl.py                          |
| CLUE_C3_ppl_56b537          | opencompass/configs/datasets/CLUE_C3/CLUE_C3_ppl_56b537.py                   |
| CLUE_C3_ppl_e24a31          | opencompass/configs/datasets/CLUE_C3/CLUE_C3_ppl_e24a31.py                   |
| CLUE_CMRC_gen               | opencompass/configs/datasets/CLUE_CMRC/CLUE_CMRC_gen.py                      |
| CLUE_CMRC_gen_1bd3c8        | opencompass/configs/datasets/CLUE_CMRC/CLUE_CMRC_gen_1bd3c8.py               |
| CLUE_CMRC_gen_3749cd        | opencompass/configs/datasets/CLUE_CMRC/CLUE_CMRC_gen_3749cd.py               |
| CLUE_CMRC_gen_8484b9        | opencompass/configs/datasets/CLUE_CMRC/CLUE_CMRC_gen_8484b9.py               |
| CLUE_CMRC_gen_941108        | opencompass/configs/datasets/CLUE_CMRC/CLUE_CMRC_gen_941108.py               |
| CLUE_DRCD_gen               | opencompass/configs/datasets/CLUE_DRCD/CLUE_DRCD_gen.py                      |
| CLUE_DRCD_gen_1bd3c8        | opencompass/configs/datasets/CLUE_DRCD/CLUE_DRCD_gen_1bd3c8.py               |
| CLUE_DRCD_gen_3749cd        | opencompass/configs/datasets/CLUE_DRCD/CLUE_DRCD_gen_3749cd.py               |
| CLUE_DRCD_gen_8484b9        | opencompass/configs/datasets/CLUE_DRCD/CLUE_DRCD_gen_8484b9.py               |
| CLUE_DRCD_gen_941108        | opencompass/configs/datasets/CLUE_DRCD/CLUE_DRCD_gen_941108.py               |
| CLUE_afqmc_gen              | opencompass/configs/datasets/CLUE_afqmc/CLUE_afqmc_gen.py                    |
| CLUE_afqmc_gen_901306       | opencompass/configs/datasets/CLUE_afqmc/CLUE_afqmc_gen_901306.py             |
| CLUE_afqmc_ppl              | opencompass/configs/datasets/CLUE_afqmc/CLUE_afqmc_ppl.py                    |
| CLUE_afqmc_ppl_378c5b       | opencompass/configs/datasets/CLUE_afqmc/CLUE_afqmc_ppl_378c5b.py             |
| CLUE_afqmc_ppl_6507d7       | opencompass/configs/datasets/CLUE_afqmc/CLUE_afqmc_ppl_6507d7.py             |
| CLUE_afqmc_ppl_7b0c1e       | opencompass/configs/datasets/CLUE_afqmc/CLUE_afqmc_ppl_7b0c1e.py             |
| CLUE_cmnli_gen              | opencompass/configs/datasets/CLUE_cmnli/CLUE_cmnli_gen.py                    |
| CLUE_cmnli_gen_1abf97       | opencompass/configs/datasets/CLUE_cmnli/CLUE_cmnli_gen_1abf97.py             |
| CLUE_cmnli_gen_51e956       | opencompass/configs/datasets/CLUE_cmnli/CLUE_cmnli_gen_51e956.py             |
| CLUE_cmnli_ppl              | opencompass/configs/datasets/CLUE_cmnli/CLUE_cmnli_ppl.py                    |
| CLUE_cmnli_ppl_98dd6e       | opencompass/configs/datasets/CLUE_cmnli/CLUE_cmnli_ppl_98dd6e.py             |
| CLUE_cmnli_ppl_ef69e7       | opencompass/configs/datasets/CLUE_cmnli/CLUE_cmnli_ppl_ef69e7.py             |
| CLUE_cmnli_ppl_fdc6de       | opencompass/configs/datasets/CLUE_cmnli/CLUE_cmnli_ppl_fdc6de.py             |
| CLUE_ocnli_gen              | opencompass/configs/datasets/CLUE_ocnli/CLUE_ocnli_gen.py                    |
| CLUE_ocnli_gen_51e956       | opencompass/configs/datasets/CLUE_ocnli/CLUE_ocnli_gen_51e956.py             |
| CLUE_ocnli_gen_c4cb6c       | opencompass/configs/datasets/CLUE_ocnli/CLUE_ocnli_gen_c4cb6c.py             |
| CLUE_ocnli_ppl              | opencompass/configs/datasets/CLUE_ocnli/CLUE_ocnli_ppl.py                    |
| CLUE_ocnli_ppl_98dd6e       | opencompass/configs/datasets/CLUE_ocnli/CLUE_ocnli_ppl_98dd6e.py             |
| CLUE_ocnli_ppl_ef69e7       | opencompass/configs/datasets/CLUE_ocnli/CLUE_ocnli_ppl_ef69e7.py             |
| CLUE_ocnli_ppl_fdc6de       | opencompass/configs/datasets/CLUE_ocnli/CLUE_ocnli_ppl_fdc6de.py             |
| FewCLUE_bustm_gen           | opencompass/configs/datasets/FewCLUE_bustm/FewCLUE_bustm_gen.py              |
| FewCLUE_bustm_gen_634f41    | opencompass/configs/datasets/FewCLUE_bustm/FewCLUE_bustm_gen_634f41.py       |
| FewCLUE_bustm_ppl           | opencompass/configs/datasets/FewCLUE_bustm/FewCLUE_bustm_ppl.py              |
| FewCLUE_bustm_ppl_4b16c0    | opencompass/configs/datasets/FewCLUE_bustm/FewCLUE_bustm_ppl_4b16c0.py       |
| FewCLUE_bustm_ppl_9ef540    | opencompass/configs/datasets/FewCLUE_bustm/FewCLUE_bustm_ppl_9ef540.py       |
| FewCLUE_bustm_ppl_e53034    | opencompass/configs/datasets/FewCLUE_bustm/FewCLUE_bustm_ppl_e53034.py       |
| FewCLUE_chid_gen            | opencompass/configs/datasets/FewCLUE_chid/FewCLUE_chid_gen.py                |
| FewCLUE_chid_gen_0a29a2     | opencompass/configs/datasets/FewCLUE_chid/FewCLUE_chid_gen_0a29a2.py         |
| FewCLUE_chid_ppl            | opencompass/configs/datasets/FewCLUE_chid/FewCLUE_chid_ppl.py                |
| FewCLUE_chid_ppl_8f2872     | opencompass/configs/datasets/FewCLUE_chid/FewCLUE_chid_ppl_8f2872.py         |
| FewCLUE_chid_ppl_acccb5     | opencompass/configs/datasets/FewCLUE_chid/FewCLUE_chid_ppl_acccb5.py         |
| FewCLUE_cluewsc_gen         | opencompass/configs/datasets/FewCLUE_cluewsc/FewCLUE_cluewsc_gen.py          |
| FewCLUE_cluewsc_gen_c68933  | opencompass/configs/datasets/FewCLUE_cluewsc/FewCLUE_cluewsc_gen_c68933.py   |
| FewCLUE_cluewsc_ppl         | opencompass/configs/datasets/FewCLUE_cluewsc/FewCLUE_cluewsc_ppl.py          |
| FewCLUE_cluewsc_ppl_12e4e0  | opencompass/configs/datasets/FewCLUE_cluewsc/FewCLUE_cluewsc_ppl_12e4e0.py   |
| FewCLUE_cluewsc_ppl_4284a0  | opencompass/configs/datasets/FewCLUE_cluewsc/FewCLUE_cluewsc_ppl_4284a0.py   |
| FewCLUE_cluewsc_ppl_868415  | opencompass/configs/datasets/FewCLUE_cluewsc/FewCLUE_cluewsc_ppl_868415.py   |
| FewCLUE_csl_gen             | opencompass/configs/datasets/FewCLUE_csl/FewCLUE_csl_gen.py                  |
| FewCLUE_csl_gen_28b223      | opencompass/configs/datasets/FewCLUE_csl/FewCLUE_csl_gen_28b223.py           |
| FewCLUE_csl_gen_87f4a8      | opencompass/configs/datasets/FewCLUE_csl/FewCLUE_csl_gen_87f4a8.py           |
| FewCLUE_csl_ppl             | opencompass/configs/datasets/FewCLUE_csl/FewCLUE_csl_ppl.py                  |
| FewCLUE_csl_ppl_769f8d      | opencompass/configs/datasets/FewCLUE_csl/FewCLUE_csl_ppl_769f8d.py           |
| FewCLUE_csl_ppl_841b62      | opencompass/configs/datasets/FewCLUE_csl/FewCLUE_csl_ppl_841b62.py           |
| FewCLUE_eprstmt_gen         | opencompass/configs/datasets/FewCLUE_eprstmt/FewCLUE_eprstmt_gen.py          |
| FewCLUE_eprstmt_gen_740ea0  | opencompass/configs/datasets/FewCLUE_eprstmt/FewCLUE_eprstmt_gen_740ea0.py   |
| FewCLUE_eprstmt_ppl         | opencompass/configs/datasets/FewCLUE_eprstmt/FewCLUE_eprstmt_ppl.py          |
| FewCLUE_eprstmt_ppl_1ce587  | opencompass/configs/datasets/FewCLUE_eprstmt/FewCLUE_eprstmt_ppl_1ce587.py   |
| FewCLUE_eprstmt_ppl_f1e631  | opencompass/configs/datasets/FewCLUE_eprstmt/FewCLUE_eprstmt_ppl_f1e631.py   |
| FewCLUE_ocnli_fc_gen        | opencompass/configs/datasets/FewCLUE_ocnli_fc/FewCLUE_ocnli_fc_gen.py        |
| FewCLUE_ocnli_fc_gen_f97a97 | opencompass/configs/datasets/FewCLUE_ocnli_fc/FewCLUE_ocnli_fc_gen_f97a97.py |
| FewCLUE_ocnli_fc_ppl        | opencompass/configs/datasets/FewCLUE_ocnli_fc/FewCLUE_ocnli_fc_ppl.py        |
| FewCLUE_ocnli_fc_ppl_9e8b3d | opencompass/configs/datasets/FewCLUE_ocnli_fc/FewCLUE_ocnli_fc_ppl_9e8b3d.py |
| FewCLUE_ocnli_fc_ppl_c08300 | opencompass/configs/datasets/FewCLUE_ocnli_fc/FewCLUE_ocnli_fc_ppl_c08300.py |
| FewCLUE_tnews_gen           | opencompass/configs/datasets/FewCLUE_tnews/FewCLUE_tnews_gen.py              |
| FewCLUE_tnews_gen_b90e4a    | opencompass/configs/datasets/FewCLUE_tnews/FewCLUE_tnews_gen_b90e4a.py       |
| FewCLUE_tnews_ppl           | opencompass/configs/datasets/FewCLUE_tnews/FewCLUE_tnews_ppl.py              |
| FewCLUE_tnews_ppl_7d1c07    | opencompass/configs/datasets/FewCLUE_tnews/FewCLUE_tnews_ppl_7d1c07.py       |
| FewCLUE_tnews_ppl_d10e8a    | opencompass/configs/datasets/FewCLUE_tnews/FewCLUE_tnews_ppl_d10e8a.py       |
| FewCLUE_tnews_ppl_fff486    | opencompass/configs/datasets/FewCLUE_tnews/FewCLUE_tnews_ppl_fff486.py       |
+-----------------------------+------------------------------------------------------------------------------+

评估过程中观察显存占用情况

<img src="./Image/2025-05-20-00-12-13.png" style="margin-left: 0" width="50%">

评估完成

<img src="./Image/2025-05-20-00-29-17.png" style="margin-left: 0" width="70%">

### 3.3.4. <a id='toc3_3_4_'></a>[评估结果](#toc0_)

/root/AI-WSL/project/projectopencompass/opencompass/outputs/default/20250519_234503/summary下有三个格式的结果文件，以txt格式结果为例

<img src="./Image/2025-05-20-00-33-53.png" style="margin-left: 0" width="80%">

从以上结果可以看出'Qwen/Qwen1.5-1.8B-Chat'结果更好。

# 4. <a id='toc4_'></a>[模型训练](#toc0_)

## 4.1. <a id='toc4_1_'></a>[模型训练框架选择：](#toc0_)

LLamaFactory还是Xtuner?  
因为当前任务的结果更偏向于主观评测，xtener就提供了在训练过程中的主观评测，LLamaFactory在训练过程中看不到主观评测，只能看到loss这种客观数据，因此选择xtuner.

## 4.2. <a id='toc4_2_'></a>[数据集格式转换](#toc0_)

原数据格式

[
  {
    "user": "闺蜜把我秘密当谈资，该不该撕破脸？",
    "assistant": "哎呀，这种事情确实很难处理呢。你觉得她的行为伤害到了你吗？💖我们可以一起想想解决的办法。",
    "style": "温柔"
  },
  {
    "user": "老妈非让我嫁给她同事儿子，怎么逃啊！",
    "assistant": "哎呀，这种事情确实挺棘手的。💖 你心里是怎么想的呢？是不是感到有些压力？我们可以一起想想解决的办法。🌸",
    "style": "温柔"
  },
  {
    "user": "同事抢功时故意提高音量，要当场揭穿吗？",
    "assistant": "哎呀，这情况确实有点棘手。你心里是怎么想的呢？觉得揭穿会不会让场面变得尴尬呢？我们可以先分析一下利弊，再决定怎么做。💖",
    "style": "温柔"
  }
]

目标格式：

[
  {
    "conversation": [
      {
        "input": "闺蜜把我秘密当谈资，该不该撕破脸？",
        "output": "温柔\n哎呀，这种事情确实很难处理呢。你觉得她的行为伤害到了你吗？💖我们可以一起想想解决的办法。"
      }
    ]
  },
  {
    "conversation": [
      {
        "input": "老妈非让我嫁给她同事儿子，怎么逃啊！",
        "output": "温柔\n哎呀，这种事情确实挺棘手的。💖 你心里是怎么想的呢？是不是感到有些压力？我们可以一起想想解决的办法。🌸"
      }
    ]
  },
  {
    "conversation": [
      {
        "input": "同事抢功时故意提高音量，要当场揭穿吗？",
        "output": "温柔\n哎呀，这情况确实有点棘手。你心里是怎么想的呢？觉得揭穿会不会让场面变得尴尬呢？我们可以先分析一下利弊，再决定怎么做。💖"
      }
    ]
  },

格式转换代码data_convert.py：

注意修改文件路径

import json

def convert_format(source_data):
    target_data = []
    for item in source_data:
        # 构建新的对话格式
        new_convo = {
            "conversation": [
                {
                    "input": item["user"],
                    "output": f"{item['style']}\n{item['assistant']}"
                }
            ]
        }
        target_data.append(new_convo)
    return target_data
# 5. <a id='toc5_'></a>[从文件读取源数据](#toc0_)
with open("D:/AI/data/style_chat_data1.json", "r", encoding="utf-8") as f:
    source_data = json.load(f)

# 6. <a id='toc6_'></a>[执行转换](#toc0_)
converted_data = convert_format(source_data)

# 7. <a id='toc7_'></a>[写入目标文件](#toc0_)
with open("output.json", "w", encoding="utf-8") as f:
    json.dump(converted_data, f, ensure_ascii=False, indent=2)

视频进度 00：42:44